# 🧠 Notebook 02 — Model Prototyping & Evaluation
**Retail Sales Neural Network Project**

This notebook:
- Runs the full preprocessing pipeline
- Trains the ANN
- Evaluates MAE, RMSE, R²
- Plots training curves and residual diagnostics

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import run_preprocessing_pipeline
from src.train_pipeline import (
    build_model, train_model, evaluate_model,
    plot_training_curves, plot_residuals,
    export_predictions_for_powerbi, get_callbacks
)

plt.style.use('dark_background')
print('Setup complete ✓')

## 1. Run Preprocessing Pipeline

In [ ]:
X_train, X_test, y_train, y_test, feature_names, scaler = run_preprocessing_pipeline(
    filepath='../data/raw/sales.csv'
)

print(f'\nTrain shape : {X_train.shape}')
print(f'Test shape  : {X_test.shape}')
print(f'\nFeatures ({len(feature_names)}):')
for i, f in enumerate(feature_names):
    print(f'  [{i:02d}] {f}')

## 2. Inspect Model Architecture

In [ ]:
model = build_model(input_dim=X_train.shape[1])
model.summary()

## 3. Train the Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Override paths for notebook context
import src.train_pipeline as tp
tp.MODELS_DIR = __import__('pathlib').Path('../models')
tp.MODEL_PATH = tp.MODELS_DIR / 'sales_model.keras'

model, history = train_model(X_train, y_train)

## 4. Training Curves

In [ ]:
plot_training_curves(history, save_path='../data/processed/training_curves.png')

## 5. Evaluate on Test Set

In [ ]:
metrics, y_pred = evaluate_model(model, X_test, y_test)

# Render as styled table
metrics_df = pd.DataFrame({
    'Metric': ['MAE ($)', 'RMSE ($)', 'R² Score'],
    'Value':  [f"${metrics['MAE']:,.2f}", f"${metrics['RMSE']:,.2f}", f"{metrics['R2']:.4f}"]
})
metrics_df.set_index('Metric')

## 6. Residual Diagnostics

In [ ]:
plot_residuals(y_test, y_pred, save_path='../data/processed/residual_diagnostics.png')

## 7. Actual vs. Predicted Sample

In [ ]:
# Display first 20 predictions vs actual
comparison = pd.DataFrame({
    'Actual ($)':    y_test[:20],
    'Predicted ($)': y_pred[:20],
    'Error ($)':     y_test[:20] - y_pred[:20],
    'Error %':       ((y_test[:20] - y_pred[:20]) / (y_test[:20] + 1e-9) * 100)
}).round(2)

comparison.style.bar(subset=['Error ($)'], align='mid', color=['#ff6b6b','#10b981'])\
    .format({'Actual ($)': '${:.2f}', 'Predicted ($)': '${:.2f}',
             'Error ($)': '${:.2f}', 'Error %': '{:.1f}%'})

## 8. Export Predictions for Power BI

In [ ]:
export_predictions_for_powerbi(
    y_test, y_pred,
    save_path='../data/processed/predictions.csv'
)
print('\n=== Notebook 02 Complete ===')
print('Next steps:')
print('  1. Open Power BI and connect to data/processed/powerbi_clean_sales.csv')
print('  2. Run Streamlit app: streamlit run app/app.py')